In [1]:
# libraries needed
import pandas as pd
import numpy as np

## load data

In [2]:
# reading data file
data = pd.read_csv("data.csv")
data.head()

   PatientID  Age Sex  ... Oldpeak  ST_Slope  HeartDisease
0          1   58   M  ...     0.0      Flat             0
1          2   53   F  ...     0.9      Flat             1
2          3   60   F  ...     0.3      Flat             0
3          4   68   M  ...     0.3        Up             1
4          5   52   M  ...     1.3      Flat             0

[5 rows x 13 columns]

## nulls

In [3]:
data.isnull().sum().sum()

np.int64(0)

## disease count

In [4]:
data["HeartDisease"].value_counts()

HeartDisease
1    461
0    457
Name: count, dtype: int64

## sex count

In [5]:
data["Sex"].value_counts()

Sex
M    625
F    293
Name: count, dtype: int64

## chol avg

In [6]:
round(data["Cholesterol"].mean(), 2)

np.float64(197.32)

## age range

In [7]:
print(data["Age"].max(), data["Age"].min())

77 28


## hr compare

In [8]:
print(round(data[data["HeartDisease"]==1]["MaxHR"].mean(), 2))
print(round(data[data["HeartDisease"]==0]["MaxHR"].mean(), 2))

133.07
136.37


## slope rate

In [9]:
data.groupby("ST_Slope")["HeartDisease"].mean()

ST_Slope
Down    0.583333
Flat    0.537344
Up      0.444149
Name: HeartDisease, dtype: float64

## drop cols

In [10]:
# these cols not needed
data = data.drop(columns=["PatientID", "RestingBP"])

## encode

In [11]:
# ml stuff
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

In [12]:
# manual map
data["Sex"] = data["Sex"].map({"M": 1, "F": 0})
data["ExerciseAngina"] = data["ExerciseAngina"].map({"Y": 1, "N": 0})
data["ST_Slope"] = data["ST_Slope"].map({"Up": 0, "Flat": 1, "Down": 2})

# label encode
cp_encoder = LabelEncoder()
data["ChestPainType"] = cp_encoder.fit_transform(data["ChestPainType"])

ecg_encoder = LabelEncoder()
data["RestingECG"] = ecg_encoder.fit_transform(data["RestingECG"])

## save encoders

In [13]:
joblib.dump(cp_encoder, "cp_encoder.joblib")
joblib.dump(ecg_encoder, "ecg_encoder.joblib")

['ecg_encoder.joblib']

## x y split

In [14]:
X = data.drop(columns="HeartDisease")
Y = data["HeartDisease"]

## try log reg

In [15]:
model1 = LogisticRegression(max_iter=1000)
model1.fit(X, Y)

LogisticRegression(max_iter=1000)

In [16]:
preds1 = model1.predict(X)
print("Accuracy:", accuracy_score(Y, preds1))

Accuracy: 0.6405228758169934


## try random forest

In [17]:
model2 = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=7)
model2.fit(X, Y)

RandomForestClassifier(max_depth=5, n_estimators=50, random_state=7)

In [18]:
preds2 = model2.predict(X)
print("Accuracy:", accuracy_score(Y, preds2))

Accuracy: 0.7287581699346405


## rf is better save it

In [19]:
joblib.dump(model2, "model.joblib")

['model.joblib']

## save data

In [20]:
data.to_csv("final_data.csv", index=False)